In [1]:
# Cell 1: Imports
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# Cell 2: Configurations
# [cite: 73] Sử dụng UPPER_SNAKE_CASE cho hằng số cấu hình
RAW_FILE_PATH = 'data_dongmai_all_raw.csv'
CLEAN_FILE_PATH = 'data_dongmai_clean.csv'

# Danh sách 13 cột đặc trưng (Features) mở rộng bao gồm Khí tượng, Khí thải và Thời gian
FEATURE_COLUMNS = [
    'temperature_2m', 'relative_humidity_2m', 'precipitation',
    'nitrogen_dioxide', 'sulphur_dioxide', 'carbon_monoxide', 'ozone', 'aerosol_optical_depth',
    'hour', 'day_of_week', 'month', 'wind_u', 'wind_v'
]

# Biến mục tiêu dự báo [cite: 51]
TARGET_COLUMN = 'pm2_5'

# Cell 3: Core Functions
def load_raw_data(file_path):
    """
    Đọc dữ liệu thô từ file CSV.
    Input: Đường dẫn file.
    Output: DataFrame.
    """
    try:
        df = pd.read_csv(file_path)
        print(f"Đã tải thành công {len(df)} dòng dữ liệu thô từ '{file_path}'.")
        return df
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {file_path}. Đảm bảo bạn đã chạy xong Bước 1.")
        return None

def handle_missing_values(df):
    """
    Xử lý khuyết thiếu bằng Forward Fill và Backward Fill.
    """
    # [cite: 54] Áp dụng phương pháp nội suy ffill và bfill do đặc thù dữ liệu thời tiết có tính kế thừa.
    missing_before = df.isnull().sum().sum()
    df_clean = df.ffill().bfill()
    missing_after = df_clean.isnull().sum().sum()
    print(f"Đã xử lý khuyết thiếu: {missing_before} -> {missing_after} giá trị null.")
    return df_clean

def extract_time_features(df):
    """
    Phân rã cột Datetime thành Giờ, Ngày trong tuần, Tháng.
    """
    # [cite: 55] Trích xuất đặc trưng (Feature Engineering) để đưa vào thuật toán.
    df['time'] = pd.to_datetime(df['time'])
    df['hour'] = df['time'].dt.hour
    df['day_of_week'] = df['time'].dt.dayofweek
    df['month'] = df['time'].dt.month
    print("Đã trích xuất xong các biến thời gian (Proxy Variables).")
    return df

def transform_wind_vectors(df):
    """
    Phân rã Hướng gió và Tốc độ gió thành vector U (Đông-Tây), V (Bắc-Nam).
    """
    # [cite: 141] Áp dụng Toán học Lượng giác để phân rã Hướng gió giúp AI không hiểu nhầm góc không gian.
    wind_dir_rad = np.radians(df['wind_direction_10m'])
    wind_speed = df['wind_speed_10m']
    
    df['wind_u'] = -wind_speed * np.sin(wind_dir_rad)
    df['wind_v'] = -wind_speed * np.cos(wind_dir_rad)
    
    # Xóa cột gốc để tránh hiện tượng đa cộng tuyến (multicollinearity)
    df = df.drop(columns=['wind_speed_10m', 'wind_direction_10m'])
    print("Đã chuyển đổi thành công vector gió U và V bằng Lượng giác.")
    return df

def scale_features(df, feature_cols):
    """
    Chuẩn hóa dữ liệu đầu vào về thang đo [0, 1].
    """
    # [cite: 56] Áp dụng thuật toán MinMaxScaler nhằm tránh hiện tượng lệch trọng số trong Học máy.
    scaler = MinMaxScaler()
    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.fit_transform(df_scaled[feature_cols])
    print("Đã chuẩn hóa không gian đặc trưng (Feature Space) về thang [0, 1].")
    
    # Trả về thêm cái scaler để có thể áp dụng cùng một phép biến đổi cho dữ liệu mới khi triển khai mô hình.
    return df_scaled, scaler

# Cell 4: Main Execution
if __name__ == "__main__":
    print("BẮT ĐẦU QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING PIPELINE)")
    
    # 1. Tải dữ liệu
    df_raw = load_raw_data(RAW_FILE_PATH)
    
    if df_raw is not None:
        # Kiểm tra và đảm bảo bài toán là "Mô phỏng" (Simulation), không dùng biến bụi quá khứ 
        cols_to_drop = [col for col in df_raw.columns if 'pm25_' in col and col != TARGET_COLUMN]
        if cols_to_drop:
            df_raw = df_raw.drop(columns=cols_to_drop)

        # 2. Xử lý khuyết thiếu [cite: 54]
        df_step1 = handle_missing_values(df_raw)
        
        # 3. Trích xuất đặc trưng thời gian [cite: 55]
        df_step2 = extract_time_features(df_step1)
        
        # 4. Biến đổi vector gió [cite: 141]
        df_step3 = transform_wind_vectors(df_step2)
        
        # 5. Chuẩn hóa dữ liệu đầu vào [cite: 56]
        df_step4, scaler_model = scale_features(df_step3, FEATURE_COLUMNS)
        
        # 6. Lọc đúng các cột cần thiết, ghép Target Variable và lưu file
        df_final = df_step4[FEATURE_COLUMNS + [TARGET_COLUMN]]
        
        try:
            df_final.to_csv(CLEAN_FILE_PATH, index=False)
            # THÊM DÒNG NÀY VÀO ĐỂ LƯU CÁI KHUÔN ÉP:
            joblib.dump(scaler_model, 'scaler.pkl')
            print(f"\nTHÀNH CÔNG! Đã lưu dữ liệu sạch vào '{CLEAN_FILE_PATH}'.")
            print(f"Kích thước bộ dữ liệu đưa vào AI: {df_final.shape[0]} dòng, {df_final.shape[1]} cột.")
            
            # Nếu chạy trên Jupyter/Colab, hiển thị bảng trực quan
            try:
                display(df_final.head())
            except NameError:
                print(df_final.head())
        except Exception as e:
            print(f"Lỗi khi lưu file: {e}")

BẮT ĐẦU QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING PIPELINE)
Đã tải thành công 28464 dòng dữ liệu thô từ 'data_dongmai_all_raw.csv'.
Đã xử lý khuyết thiếu: 0 -> 0 giá trị null.
Đã trích xuất xong các biến thời gian (Proxy Variables).
Đã chuyển đổi thành công vector gió U và V bằng Lượng giác.
Đã chuẩn hóa không gian đặc trưng (Feature Space) về thang [0, 1].

THÀNH CÔNG! Đã lưu dữ liệu sạch vào 'data_dongmai_clean.csv'.
Kích thước bộ dữ liệu đưa vào AI: 28464 dòng, 14 cột.


,temperature_2m,relative_humidity_2m,precipitation,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide,ozone,aerosol_optical_depth,hour,day_of_week,month,wind_u,wind_v,pm2_5
0,0.179153,0.746835,0.0,0.052738,0.208226,0.164866,0.222581,0.085158,0.000000,1.0,0.0,0.457055,0.486237,37.3
1,0.172638,0.734177,0.0,0.050710,0.205656,0.161161,0.209677,0.075426,0.043478,1.0,0.0,0.445996,0.495630,35.2
2,0.162866,0.746835,0.0,0.051724,0.202228,0.160235,0.196774,0.070560,0.086957,1.0,0.0,0.456987,0.495099,33.6
3,0.146580,0.784810,0.0,0.053753,0.201371,0.160852,0.180645,0.068127,0.130435,1.0,0.0,0.481406,0.483672,32.6
4,0.133550,0.797468,0.0,0.058824,0.198800,0.161470,0.164516,0.068127,0.173913,1.0,0.0,0.504621,0.481721,32.1
